# Case Setup (`Case` object) & Coupling

Once your grids are ready (see [Grids](grids.ipynb) and [Bathymetry](bathymetry.ipynb)),
the next step is creating a `Case` — CrocoDash's orchestration object that ties your
grid definition into CESM. `Case(...)` also decides, from your chosen `compset`, which
additional forcings you're on the hook for in [Configure Forcings](configure_forcings.ipynb).

This notebook covers:
- [Section 1](#section-1-create-a-standalone-ocean-case) — the standalone-ocean case (`Case(...)` basics, compset reference table)
- [Section 2](#section-2-discover-what-a-compset-requires) — how to discover what a compset requires, *before* calling `configure_forcings`
- [Section 3](#section-3-coupling-sea-ice-cice6) — coupling: sea ice (CICE6)
- [Section 4](#section-4-coupling-biogeochemistry-marbl) — coupling: biogeochemistry (MARBL)

📖 [CrocoDash case setup docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/2_case_setup.html)

## Section 1: Create a Standalone-Ocean Case

After generating the MOM6 domain, the next step is to create a CESM case using
CrocoDash. This process is straightforward and involves instantiating the CrocoDash
`Case` object. The `Case` object requires the following inputs:

 - CESM Source Directory: A local path to a compatible CESM source copy.
 - Case Name: A unique name for the CESM case.
 - Input Directory: The directory where all necessary input files will be written.
 - MOM6 Domain Objects: the `Grid`, `Topo`, and `VGrid` created in [Grids](grids.ipynb) and [Bathymetry](bathymetry.ipynb).
 - Project ID: (Optional) A project ID, if required by the machine.
 - Compset: The set of models to be used in the Case. Standalone Ocean, Ocean-BGC, Ocean-Seaice, Ocean-Runoff.

Begin by specifying the case name and the necessary directory paths. Ensure the CESM
root directory points to your own local copy of CESM.

In [ ]:
# CESM case (experiment) name
casename = "panama-not"

# CESM source root (Update this path accordingly!!!)
cesmroot = "<CESM>"

# Place where all your input files go
inputdir = "<inputdir>"

# CESM case directory
caseroot = "<casedir>"

### Compset quick-reference

The `compset` argument controls which model components are active. Start with the
ocean-only default and swap or add stubs as needed:

| What you want | Change in `compset=` |
|---|---|
| Ocean only (default) | `compset="CR_JRA"` |
| + Sea ice (CICE6) | change `CR_JRA` → `GR_JRA` |
| + Biogeochemistry (MARBL) | add `%MARBL-BIO` to the MOM6 component, e.g. `MOM6%REGIONAL%MARBL-BIO` |
| + Data runoff (GLOFAS) | use `CR_JRA_GLOFAS` |
| + Data runoff (JRA) | use `CR_JRA` with `DROF%JRA` (see [Configure Forcings, Section 5](configure_forcings.ipynb#section-5-runoff)) |

For a deeper dive into each option:
- **CICE or BGC**: [Sections 3–4 below](#section-3-coupling-sea-ice-cice6)
- **Runoff (GLOFAS / JRA / custom)**: [Configure Forcings, Section 5](configure_forcings.ipynb#section-5-runoff)

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = 'CESM0030',
    override = True,
    machine = "derecho",
    compset = "GR_JRA" # This is the alias of the compset, the longname (which is printed when you run this command) is 1850_DATM%JRA_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_SWAV. Feel free to use either way!
)

## Section 2: Discover What a Compset Requires

Right after `Case(...)` succeeds, it prints a summary of any **required** forcing
configurations you still owe — these become your keyword arguments to
`case.configure_forcings(...)` in the next step. For a compset with MARBL BGC and
GLOFAS runoff, that printout looks like:

```
The following additional configuration options are required to run and must be
provided with any listed arguments in configure_forcings:
  - BGC: no arguments
  - BGCIC: marbl_ic_filepath
  - Runoff: no arguments
```

You don't have to build a real `Case` to see this — the same registry the constructor
calls is available standalone, keyed off any compset **long name** string:

In [ ]:
from CrocoDash.forcing_configurations import ForcingConfigRegistry


def print_required(compset):
    print(compset)
    for config_class in ForcingConfigRegistry.find_required_configurators(compset):
        user_args = ForcingConfigRegistry.get_user_args(config_class)
        print(f"  Required: {config_class.name}  (needs: {user_args or 'no arguments'})")


# Section 4 below builds a Case with this compset (MARBL, no active runoff):
print_required("1850_DATM%NYF_SLND_SICE_MOM6%MARBL-BIO%REGIONAL_SROF_SGLC_SWAV")

# Swapping in GLOFAS runoff changes the required list — a Runoff configurator joins BGC/BGCIC:
print_required("1850_DATM%NYF_SLND_SICE_MOM6%MARBL-BIO%REGIONAL_DROF%GLOFAS_SGLC_SWAV")

On a live `Case`, `case.compset_lname` gives you the resolved long name CrocoDash
already validated against, so you can run the same check without retyping it:

```python
required = ForcingConfigRegistry.find_required_configurators(case.compset_lname)
```

See [Configure Forcings](configure_forcings.ipynb) for the rest of the introspection API
(`find_valid_configurators`, `return_missing_inputs`, `validate_compset_compatibility`)
and how each required configurator maps onto a `configure_forcings()` keyword.

## Section 3: Coupling — Sea Ice (CICE6)

Swap the stub `SICE` (stub sea ice) for `CICE` in the compset. Everything else — grid,
topo, vgrid, forcings — is identical to the standalone run.

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="NCGD0011",
    override=True,
    machine="derecho",
    compset="GR_JRA",  # GR_JRA = 1850_DATM%JRA_SLND_CICE_MOM6%REGIONAL_SROF_SGLC_SWAV
)

### Optional: Warm start from CICE restart files

After a first run you can use CICE restart files as the ice initial condition instead
of the default (open water). This gives a more realistic sea-ice state at the start of
subsequent runs.

1. Find restart files in your archive: `<caseroot>/archive/rest/<year>/` — look for `*.cice.r.*.nc`.
   If unsure of the archive path, run `./xmlquery DOUT_S_ROOT` in the case directory.
2. Copy the file to your run directory: `cp <restart_file> <run_dir>/`.
3. Open `user_nl_cice` and set `ice_ic = "<restart_filename>"`.

```{note}
History files (`.h` / `.h1`) **cannot** be used as CICE initial conditions — only `.r` restart files.
```

## Section 4: Coupling — Biogeochemistry (MARBL)

MARBL (the Marine Biogeochemistry Library) is bundled with CESM. Activating it requires:

1. A compset that includes `MOM6%REGIONAL%MARBL-BIO`.
2. A MARBL global initial-condition file.
3. Passing BGC-specific kwargs to `configure_forcings` (see [Configure Forcings, Section 6](configure_forcings.ipynb#section-6-biogeochemistry-marbl)).

Optionally, combine with river nutrients by also enabling GLOFAS runoff.

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="NCGD0011",
    override=True,
    machine="derecho",
    compset="CR1850MARBL_JRA",
)

Once this `Case` exists, rerun Section 2's discovery check against `case.compset_lname`
to confirm exactly which `configure_forcings` kwargs you owe — for this compset that's
`marbl_ic_filepath` (matching the first `print_required(...)` call above, and the printed
report `Case(...)` already gave you).

## Next steps

Continue to [Configure Forcings](configure_forcings.ipynb) to fill in the BGC/runoff
arguments this compset requires, or jump to [Process Forcings](process_forcings.ipynb)
once configured.